# 01 - Exploration des données

**Projet :** Cadre de gouvernance et qualité des données - Hydro-Québec

**Auteur :** Anthony MISSE

**Date :** 2026-05-10

L'objectif est de comprendre la structure, les types, les volumes et les premières anomalies potentielles.

## 1. Import des bibliothèques

In [39]:
from pathlib import Path

import pandas as pd

DATA_RAW_DIR = Path("../data/raw")
REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Table hq_demande_electricite

### 2.1 Chargement brut

In [40]:
df_dem = pd.read_csv(DATA_RAW_DIR / "hq_demande_electricite_raw.csv", encoding="utf-8")
df_dem.head()

,date,demande (MW)
0,2023-01-01T03:00:00-05:00,20010.46
1,2023-01-01T06:00:00-05:00,20411.55
2,2023-01-01T08:00:00-05:00,21561.22
3,2023-01-01T11:00:00-05:00,22842.33
4,2023-01-01T12:00:00-05:00,23224.36


### 2.2 Dimensions et types

In [41]:
print("Dimensions :", df_dem.shape)
df_dem.dtypes

Dimensions : (43824, 2)


date             object
demande (MW)    float64
dtype: object

> **OBS-001** - La colonne `date` est de type `object` (string). Le format observé est ISO 8601 avec fuseau horaire
> (`2023-01-01T03:00:00-05:00`). Une conversion en `datetime2` sera nécessaire dans le pipeline ADF.

> **OBS-002** - La colonne `demande (MW)` est de type `float64` - pandas a réussi à l'inférer directement.  
> Cependant son nom contient un espace et des parenthèses, ce qui causera des problèmes dans ADF et en SQL.  
> Renommage prévu : `demande_mw`.

### 2.3 Valeurs manquantes

In [42]:
df_dem.isnull().sum()

date             0
demande (MW)    45
dtype: int64

> **OBS-003** - **45 valeurs nulles** dans `demande (MW)`, soit **0.10 %** des 43 824 lignes.
> Ce taux est inférieur au seuil de blocage fixé à 1 %. Stratégie retenue : interpolation linéaire  
> sur les valeurs adjacentes (applicable car les nulls ne sont pas consécutifs sur plus de 3 créneaux).

### 2.4 Doublons sur la colonne date

In [43]:
nb_doublons = df_dem.duplicated(subset=["date"]).sum()
print("Nombre de doublons sur date :", nb_doublons)

Nombre de doublons sur date : 6


On affiche les lignes concernées pour en comprendre l'origine.

In [44]:
df_dem[df_dem.duplicated(subset=["date"], keep=False)].sort_values("date")

,date,demande (MW)
22954,2019-11-03T01:00:00-05:00,18301.79
22955,2019-11-03T01:00:00-05:00,17963.20
11351,2020-11-01T01:00:00-05:00,20001.77
25860,2020-11-01T01:00:00-05:00,19884.29
28759,2021-11-07T01:00:00-05:00,18793.52
38327,2021-11-07T01:00:00-05:00,18685.38
8169,2022-11-06T01:00:00-05:00,14788.72
8170,2022-11-06T01:00:00-05:00,14447.54
5677,2023-01-01T00:00:00-05:00,22945.84
43551,2023-01-01T00:00:00-05:00,20516.80


> **OBS-004** - Les 6 doublons correspondent aux nuits du passage de l'heure avancée à l'heure normale
> (premier dimanche de novembre au Québec), plus une entrée au 1er janvier 2023.
> Ces doublons ne sont pas des erreurs de saisie - ils reflètent la répétition réelle d'un créneau horaire UTC.
> Stratégie retenue : conservation de la première occurrence (`keep='first'`).
> Nombre de lignes après correction : **43 818**.

### 2.5 Distribution de demande (MW)

In [45]:
df_dem["demande (MW)"].describe()

count    43779.000000
mean     21544.832961
std       5193.772072
min      13293.750000
25%      17705.145000
50%      19810.260000
75%      25425.325000
max      42472.830000
Name: demande (MW), dtype: float64

On applique la méthode IQR pour identifier les valeurs potentiellement aberrantes.

In [46]:
Q1 = df_dem["demande (MW)"].quantile(0.25)
Q3 = df_dem["demande (MW)"].quantile(0.75)
IQR = Q3 - Q1
seuil_haut = Q3 + 1.5 * IQR
seuil_bas = Q1 - 1.5 * IQR

print(f"Seuil IQR bas   : {seuil_bas:.2f} MW")
print(f"Seuil IQR haut  : {seuil_haut:.2f} MW")
print(f'Valeurs > seuil haut : {(df_dem["demande (MW)"] > seuil_haut).sum()}')
print(f'Valeurs < seuil bas  : {(df_dem["demande (MW)"] < seuil_bas).sum()}')

Seuil IQR bas   : 6124.88 MW
Seuil IQR haut  : 37005.60 MW
Valeurs > seuil haut : 94
Valeurs < seuil bas  : 0


On affiche un échantillon des lignes au-dessus du seuil pour comprendre le contexte.

In [47]:
masque_ext = df_dem["demande (MW)"] > seuil_haut
df_dem[masque_ext][["date", "demande (MW)"]].head(10)

,date,demande (MW)
233,2023-02-03T09:00:00-05:00,37528.23
234,2023-02-03T13:00:00-05:00,37334.31
235,2023-02-03T14:00:00-05:00,37861.89
236,2023-02-03T17:00:00-05:00,40585.14
237,2023-02-04T03:00:00-05:00,39410.28
238,2023-02-04T05:00:00-05:00,39911.76
240,2023-02-04T19:00:00-05:00,39317.89
5772,2022-01-11T07:00:00-05:00,37844.73
5776,2022-01-11T17:00:00-05:00,38227.06
5777,2022-01-11T20:00:00-05:00,38976.50


> **OBS-005** - Les 94 valeurs dépassant le seuil IQR de 37 005.60 MW sont concentrées en
> **janvier-février 2022 et 2023**, correspondant à des vagues de grand froid au Québec.  
> Ces valeurs sont **légitimes** - elles ne doivent pas être supprimées ni corrigées.  
> Elles seront documentées comme cas exceptionnels dans le dictionnaire de données et les règles de qualité (RQ-04b).

## 3. Table hq_consommation_secteur

### 3.1 Chargement brut

In [48]:
df_sec = pd.read_csv(DATA_RAW_DIR / "hq_consommation_secteur_raw.csv", encoding="utf-8")
df_sec.head()

,REGION_ADM_QC_TXT,ANNEE_MOIS,SECTEUR,Total (kWh)
0,Abitibi-Témiscamingue,2016-01-01,INDUSTRIEL,3.152834e+08
1,Abitibi-Témiscamingue,2016-02-01,AGRICOLE,3.440854e+06
2,Abitibi-Témiscamingue,2016-02-01,INDUSTRIEL,3.099185e+08
3,Abitibi-Témiscamingue,2016-04-01,COMMERCIAL,4.213748e+07
4,Abitibi-Témiscamingue,2016-04-01,INSTITUTIONNEL,2.314361e+07


### 3.2 Dimensions et types

In [49]:
print("Dimensions :", df_sec.shape)
df_sec.dtypes

Dimensions : (10200, 4)


REGION_ADM_QC_TXT     object
ANNEE_MOIS            object
SECTEUR               object
Total (kWh)          float64
dtype: object

> **OBS-006** - `ANNEE_MOIS` est de type `object`. Format observé : `yyyy-MM-dd`.
> Conversion en type `date` prévue dans le pipeline ADF (`toDate(ANNEE_MOIS, 'yyyy-MM-dd')`).  

> **OBS-007** - `Total (kWh)` est déjà de type `float64` - pandas l'a inféré correctement.  
> Le nom contient un espace et des parenthèses. Renommage prévu : `total_kwh`.

### 3.3 Valeurs manquantes

In [50]:
df_sec.isnull().sum()

REGION_ADM_QC_TXT    0
ANNEE_MOIS           0
SECTEUR              0
Total (kWh)          0
dtype: int64

0 valeur nulle dans toutes les colonnes.

### 3.4 Valeurs distinctes de SECTEUR

In [51]:
df_sec["SECTEUR"].value_counts()

INDUSTRIEL        2040
AGRICOLE          2040
COMMERCIAL        2040
INSTITUTIONNEL    2040
RÉSIDENTIEL       2040
Name: SECTEUR, dtype: int64

> **OBS-008** - Les 5 secteurs ont exactement **2040 occurrences chacun** (17 régions × 120 mois).
> La distribution est parfaitement équilibrée - aucune région ni aucun mois ne manque.  
> Les valeurs sont toutes en majuscules. Normalisation prévue dans ADF : `initCap(lower(SECTEUR))`.  
> Résultat attendu : Agricole, Commercial, Industriel, Institutionnel, Résidentiel.

### 3.5 Régions administratives

In [52]:
print("Nombre de régions distinctes :", df_sec["REGION_ADM_QC_TXT"].nunique())
df_sec["REGION_ADM_QC_TXT"].unique()

Nombre de régions distinctes : 17


array(['Abitibi-Témiscamingue', 'Bas-Saint-Laurent', 'Capitale-Nationale',
       'Centre-du-Québec', 'Chaudière-Appalaches', 'Côte-Nord', 'Estrie',
       'Gaspésie--Îles-de-la-Madeleine', 'Lanaudière', 'Laurentides',
       'Laval', 'Mauricie', 'Montréal', 'Montérégie', 'Nord-du-Québec',
       'Outaouais', 'Saguenay--Lac-Saint-Jean'], dtype=object)

> Les 17 régions correspondent aux régions administratives officielles du Québec. Aucune valeur inconnue.

### 3.6 Distribution de Total (kWh)

In [53]:
df_sec["Total (kWh)"].describe()

count    1.020000e+04
mean     1.723920e+08
std      2.634085e+08
min      1.041400e+04
25%      2.377418e+07
50%      6.770911e+07
75%      2.046468e+08
max      2.393576e+09
Name: Total (kWh), dtype: float64

In [54]:
print("Valeurs nulles    :", df_sec["Total (kWh)"].isnull().sum())
print("Valeurs <= 0      :", (df_sec["Total (kWh)"] <= 0).sum())

Valeurs nulles    : 0
Valeurs <= 0      : 0


> Toutes les valeurs de `Total (kWh)` sont strictement positives. La valeur maximale (2.39 TWh)  
> correspond à une région industrielle intense sur un mois de forte activité - cohérent avec le contexte HQ.

### 3.7 Unicité de la clé composite (REGION_ADM_QC_TXT, ANNEE_MOIS, SECTEUR)

In [55]:
nb_doublons_sec = df_sec.duplicated(
    subset=["REGION_ADM_QC_TXT", "ANNEE_MOIS", "SECTEUR"]
).sum()
print("Doublons sur la clé composite :", nb_doublons_sec)

Doublons sur la clé composite : 0


## 4. Cohérence temporelle entre les deux tables

In [56]:
date_dem = pd.to_datetime(df_dem["date"], errors="coerce", utc=True)
date_sec = pd.to_datetime(df_sec["ANNEE_MOIS"], errors="coerce")

print("hq_demande - debut:", date_dem.min(), "| fin:", date_dem.max())
print("hq_secteur - debut:", date_sec.min(), "| fin:", date_sec.max())

hq_demande - debut: 2019-01-01 06:00:00+00:00 | fin: 2024-01-01 05:00:00+00:00
hq_secteur - debut: 2016-01-01 00:00:00 | fin: 2025-12-01 00:00:00


> **OBS-009** - Les deux tables couvrent des périodes différentes :
> - `hq_demande` : 2019 à 2024 (5 ans, granularité horaire)
> - `hq_secteur` : 2016 à 2025 (10 ans, granularité mensuelle)
>
> **Période de chevauchement : 2019-01-01 à 2024-01-01.**  
> Toute analyse croisée doit se limiter à cette fenêtre commune.  
> Ce filtre devra être appliqué dans les requêtes SQL et les mesures DAX de Power BI.

## 5. Export des profils de qualité

In [57]:
def basic_profile(df, name):
    total = len(df)
    rows = []
    for col in df.columns:
        nulls = int(df[col].isna().sum())
        rows.append(
            {
                "table": name,
                "colonne": col,
                "dtype": str(df[col].dtype),
                "nb_lignes": total,
                "nb_non_null": total - nulls,
                "nb_null": nulls,
                "pct_null": round(nulls * 100.0 / total, 2) if total > 0 else 0.0,
                "nb_distinct": int(df[col].nunique(dropna=True)),
            }
        )
    return pd.DataFrame(rows)


profil_dem = basic_profile(df_dem, "hq_demande_electricite")
profil_sec = basic_profile(df_sec, "hq_consommation_secteur")

profil_dem.to_csv(REPORTS_DIR / "profil_hq_demande.csv", index=False)
profil_sec.to_csv(REPORTS_DIR / "profil_hq_secteur.csv", index=False)

print("Profils sauvegardes dans reports/")
print("")

print("Profil Demande electricité")
print(profil_dem)
print("")

print("Profil Consommation secteur")
print(profil_sec)

Profils sauvegardes dans reports/

Profil Demande electricité
                    table       colonne    dtype  nb_lignes  nb_non_null  \
0  hq_demande_electricite          date   object      43824        43824   
1  hq_demande_electricite  demande (MW)  float64      43824        43779   

   nb_null  pct_null  nb_distinct  
0        0       0.0        43818  
1       45       0.1        43124  

Profil Consommation secteur
                     table            colonne    dtype  nb_lignes  \
0  hq_consommation_secteur  REGION_ADM_QC_TXT   object      10200   
1  hq_consommation_secteur         ANNEE_MOIS   object      10200   
2  hq_consommation_secteur            SECTEUR   object      10200   
3  hq_consommation_secteur        Total (kWh)  float64      10200   

   nb_non_null  nb_null  pct_null  nb_distinct  
0        10200        0       0.0           17  
1        10200        0       0.0          120  
2        10200        0       0.0            5  
3        10200        0       

## 6. Synthèse des observations

Ces observations émergent de l'exploration et servent de base à `regles_qualite.md` et `dictionnaire_donnees.md`.

| Code    | Table      | Colonne      | Observation                                                                                           |
| ------- | ---------- | ------------ | ----------------------------------------------------------------------------------------------------- |
| OBS-001 | hq_demande | date         | Type object, format ISO 8601 avec fuseau horaire (-05:00). Conversion datetime2 dans ADF.             |
| OBS-002 | hq_demande | demande (MW) | Nom avec espace et parenthèses. Renommage en demande_mw prévu.                                        |
| OBS-003 | hq_demande | demande (MW) | 45 valeurs nulles (0.10 %). Interpolation linéaire sous le seuil de 1 %.                              |
| OBS-004 | hq_demande | date         | 6 doublons liés au changement d'heure (1er dimanche de novembre). Conservation de la 1ère occurrence. |
| OBS-005 | hq_demande | demande (MW) | 94 valeurs > 37 005.60 MW en jan-fév 2022 et 2023 - pics de grand froid, légitimes.                   |
| OBS-006 | hq_secteur | ANNEE_MOIS   | Type object, format yyyy-MM-dd. Conversion en date dans ADF.                                          |
| OBS-007 | hq_secteur | Total (kWh)  | Nom avec espace et parenthèses. Renommage en total_kwh prévu.                                         |
| OBS-008 | hq_secteur | SECTEUR      | Valeurs en majuscules (AGRICOLE, etc.). Normalisation initCap(lower()) dans ADF.                      |
| OBS-009 | Les deux   | date         | Période de chevauchement : 2019-01-01 à 2024-01-01. Filtre requis dans SQL et DAX.                    |